In [ ]:
#| default_exp aio

# Async helpers

> Bridging async and sync code: `run_sync`, `iter_sync`, `ctx_sync`, `maybe_await`, and `then`, plus `Debounce` for coalescing bursts of calls


In [ ]:
#| export
import asyncio,threading
from contextlib import contextmanager
from functools import wraps
from fastcore.imports import *
from fastcore.xtras import UNSET
from fastcore.basics import *

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
import contextlib

## Calling async code from sync code

Async libraries return coroutines, which normally force every caller up the stack to become async too. The helpers below let ordinary sync code drive them instead, sharing a single event loop that runs on a background daemon thread, created on first use.

In [ ]:
#| export
_loop,_loop_lock = None,threading.Lock()

def _get_loop():
    global _loop
    with _loop_lock:
        if _loop is None:
            _loop = asyncio.new_event_loop()
            threading.Thread(target=_loop.run_forever, daemon=True, name='run_sync').start()
    return _loop

def run_sync(coro):
    "Run coroutine `coro` to completion from sync code and return its result"
    loop = _get_loop()
    try: reentrant = asyncio.get_running_loop() is loop
    except RuntimeError: reentrant = False
    if reentrant:
        coro.close()
        raise RuntimeError('run_sync called from its own event loop; use `await` instead')
    fut = asyncio.run_coroutine_threadsafe(coro, loop)
    try: return fut.result()
    except KeyboardInterrupt:
        fut.cancel()
        raise

Exceptions raised inside the coroutine propagate to the caller, and Ctrl-C cancels it rather than leaving it running on the loop thread. Calling `run_sync` from a coroutine it is already running would deadlock, so that raises `RuntimeError` instead.

In [ ]:
async def _double(x):
    await asyncio.sleep(0.01)
    return x*2

test_eq(run_sync(_double(3)), 6)

async def _boom(): raise ValueError('boom')
with expect_fail(Exception, 'boom'): run_sync(_boom())

async def _nested(): return run_sync(_double(1))
with expect_fail(Exception, 'its own event loop'): run_sync(_nested())

Because the loop lives on its own thread, `run_sync` also works where the calling thread already has a running loop, such as a Jupyter cell (where `asyncio.run` raises `RuntimeError`). Here we simulate that by calling it from inside a coroutine:

In [ ]:
async def _outer(): return run_sync(_double(4))
test_eq(await _outer(), 8)

In [ ]:
#| export
def iter_sync(agen):
    "Iterate async generator `agen` from sync code"
    try:
        while True: yield run_sync(agen.__anext__())
    except StopAsyncIteration: pass
    finally: run_sync(agen.aclose())

`iter_sync` pulls each item by running `__anext__` on the shared loop, and closes `agen` when iteration ends, including when the consumer stops early:

In [ ]:
done = []
async def _agen():
    try:
        i = 0
        while True:
            yield i
            i += 1
    finally: done.append(True)

it = iter_sync(_agen())
test_eq([next(it) for _ in range(3)], [0,1,2])
it.close()
test_eq(done, [True])

In [ ]:
#| export
@contextmanager
def ctx_sync(acm):
    "Use async context manager `acm` in a plain `with` block"
    res = run_sync(acm.__aenter__())
    try: yield res
    except BaseException as e:
        if not run_sync(acm.__aexit__(type(e), e, e.__traceback__)): raise
    else: run_sync(acm.__aexit__(None,None,None))

`__aenter__` and `__aexit__` run on the shared loop, and exception suppression behaves as it would under `async with`:

In [ ]:
events = []
@contextlib.asynccontextmanager
async def _actx():
    events.append('enter')
    try: yield 'ready'
    finally: events.append('exit')

with ctx_sync(_actx()) as v: test_eq(v, 'ready')
test_eq(events, ['enter','exit'])

In [ ]:
#| export
async def maybe_await(o):
    "Await `o` if needed, and return it"
    from inspect import isawaitable
    return await o if isawaitable(o) else o

In [ ]:
async def _f(): return 42
test_eq(await maybe_await(_f()), 42)
test_eq(await maybe_await('hello'), 'hello')

In [ ]:
#| export
def then(x, *fs):
    "Pipe `x` through each of `fs`, awaiting values as needed; result is awaitable only if `x` or a step result is"
    from inspect import isawaitable
    async def _go(x, fs):
        x = await x
        for f in fs: x = await maybe_await(f(x))
        return x
    for i,f in enumerate(fs):
        if isawaitable(x): return _go(x, fs[i:])
        x = f(x)
    return _go(x, ()) if isawaitable(x) else x

`then` lets one function body serve sync and async callers alike. It applies each of `fs` in turn, awaiting anything awaitable along the way (the starting value or any step's result), and the caller gets back a plain value if the whole chain was sync, or an awaitable otherwise. This is what a convenience method on a client offering both sync and async modes needs: written as `return then(self.gists.get(gid), ~Self.files.values(), first)`, a sync client returns the file directly while an async client returns something to `await`, with no duplicated method. Because step results are awaited as part of the chain, steps may themselves be async calls; the flip side is that a step cannot return an awaitable *as* its value.

In [ ]:
async def _f(): return dict(a=1,b=2)
test_eq(then(dict(a=1,b=2), ~Self.values(), first), 1)  # all sync: plain result
test_eq(await then(_f(), ~Self.values(), first), 1)     # awaitable start
async def _double(x): return x*2
test_eq(await then(3, _double, mul(10)), 60)            # awaitable appears mid-chain
test_eq(then(3), 3)
test_eq(await then(_f()), dict(a=1,b=2))                # no steps: pure passthrough

In [ ]:
#| export
def acache(f):
    "Cache results of async function `f`"
    cache = {}
    @wraps(f)
    async def _inner(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache: cache[key] = await f(*args, **kwargs)
        return cache[key]
    _inner.cache_clear = cache.clear
    return _inner

This is like `functools.cache` but for async functions, e.g:

In [ ]:
n = 0

@acache
async def f(x):
    global n
    n += 1
    return x*2

test_eq([await f(3), await f(3)], [6,6])
test_eq(n, 1)

In [ ]:
test_eq([await f(4), await f(4)], [8,8])
test_eq(n, 2)

In [ ]:
#| export
class CachedAwaitable:
    "Cache the result from an awaitable"
    def __init__(self, o): self.o,self.value = o,UNSET
    def __await__(self):
        if self.value is UNSET: self.value = yield from self.o.__await__()
        return self.value

In [ ]:
#| export
def reawaitable(func:callable):
    "Wraps the result of an asynchronous function into an object which can be awaited more than once"
    @wraps(func)
    def _f(*args, **kwargs): return CachedAwaitable(func(*args, **kwargs))
    return _f

`CachedCoro` and `reawaitable` are partly based on [python issue tracker](https://bugs.python.org/issue46622) code from Serhiy Storchaka. They allow an awaitable to be called multiple times.

In [ ]:
@reawaitable
async def fetch_data():
    await asyncio.sleep(0.1)
    return "data"

r = fetch_data()
print(await r)  # "data"
print(await r)  # "data" (no delay)

data


data


In [ ]:
#| export
def is_async_callable(obj):
    "Check if `obj` is an async callable, handling `partial` wrappers and callable instances"
    # Implementation from Starlette; Copyright © 2018, Encode OSS Ltd.
    from asyncio import iscoroutinefunction
    while isinstance(obj, partial): obj = obj.func
    return iscoroutinefunction(obj) or (callable(obj) and iscoroutinefunction(obj.__call__))

`is_async_callable` detects whether an object can be called asynchronously. It goes beyond `asyncio.iscoroutinefunction` by also handling `functools.partial`-wrapped async functions (unwrapping through any number of layers) and callable objects whose `__call__` method is a coroutine. The implementation is thanks to [Starlette](https://github.com/encode/starlette).

In [ ]:
async def f(): pass
assert is_async_callable(f)
assert is_async_callable(partial(f))
assert not is_async_callable(lambda: None)

In [ ]:
class AsyncObj:
    async def __call__(self): pass

class SyncObj:
    def __call__(self): pass

assert is_async_callable(AsyncObj())
assert not is_async_callable(SyncObj())

In [ ]:
#| export
async def to_aiter(items):
    "Async yield each item in `items` with `asyncio.sleep(0)` between"
    import asyncio
    for item in items:
        await asyncio.sleep(0)
        yield item

In [ ]:
test_eq([o async for o in to_aiter([10,20,30])], [10,20,30])
test_eq([o async for o in to_aiter([])], [])

In [ ]:
#| export
def maybe_aiter(items):
    "If `items` already async, return it; otherwise to_aiter"
    return items if hasattr(items, '__aiter__') else to_aiter(items)

In [ ]:
async def _agen():
    for i in [1,2,3]: yield i

ag = _agen()
test_eq([o async for o in maybe_aiter(ag)], [1,2,3])
test_eq([o async for o in maybe_aiter([1,2,3])], [1,2,3])

In [ ]:
#| export
async def mapa(f, items):
    "Async `map`; apply `f` (sync or async) to `items` (sync or async iter) concurrently via `gather`"
    from asyncio import gather
    return await gather(*[maybe_await(f(o)) async for o in maybe_aiter(items)])

In [ ]:
async def _double(x): return x*2

test_eq(await mapa(mul(2), [1,2,3]), [2,4,6])       # sync f, sync items
test_eq(await mapa(_double, [1,2,3]), [2,4,6])              # async f, sync items
test_eq(await mapa(mul(2), _agen()), [2,4,6])        # sync f, async items
test_eq(await mapa(_double, _agen()), [2,4,6])              # async f, async items

In [ ]:
#| export
async def noopa(x=None, *args, **kwargs):
    "Do nothing (async)"
    return x

## Debounce and throttle

Coalesce bursts of calls into fewer invocations of a function. A typical use is auto-saving shortly after edits stop, rather than on every keystroke.


In [ ]:
#| export
class Debounce:
    "Coalesce calls to `f` into one, made `wait` secs after calls stop (or `max_wait` secs after they start)"
    def __init__(self, f, wait, max_wait=None, leading=False, trailing=True):
        assert leading or trailing,"one of `leading`/`trailing` must be set"
        store_attr()
        self._task,self._q = None,None
        try: self._loop = asyncio.get_running_loop()
        except RuntimeError: self._loop = None

    def _put_latest(self, q, call):
        if q.full(): q.get_nowait()
        q.put_nowait(call)

    def __call__(self, *args, **kw):
        try: loop = asyncio.get_running_loop()
        except RuntimeError:
            if self._loop is None: raise RuntimeError('Debounce must first be called from an async event loop') from None
            return self._loop.call_soon_threadsafe(partial(self.__call__, *args, **kw))

        if self._loop is None: self._loop = loop
        elif loop is not self._loop: return self._loop.call_soon_threadsafe(partial(self.__call__, *args, **kw))

        self._deadline = self._loop.time()+self.wait
        call = (args, kw)
        if self._task and not self._task.done(): self._put_latest(self._q, call)
        else:
            self._q = asyncio.Queue(maxsize=1)
            self._cap = self._loop.time()+self.max_wait if self.max_wait else float('inf')
            if not self.leading: self._put_latest(self._q, call)
            self._task = self._loop.create_task(self._run(self._q, call if self.leading else None))

    async def _fire(self, p): await maybe_await(self.f(*p[0], **p[1]))
    async def _run(self, q, first):
        if first is not None: await self._fire(first)

        while True:
            delay = min(self._deadline,self._cap) - self._loop.time()
            if delay>0: await asyncio.sleep(delay); continue
            if not self.trailing:
                if not q.empty(): q.get_nowait()  # drop the rest of the burst
                return
            if q.empty(): return
            await self._fire(q.get_nowait())
            if q.empty(): return
            if self.max_wait: self._cap = self._loop.time()+self.max_wait

    def cancel(self):
        "Discard any pending call"
        if self._task and not self._task.done(): self._task.cancel()
        self._task,self._q = None,None

    async def flush(self):
        "Run any pending call now instead of waiting"
        p = self._q.get_nowait() if self._q and not self._q.empty() else None
        self.cancel()
        if p is not None: await self._fire(p)

    def __set_name__(self, owner, name): self._name = name
    def __get__(self, obj, objtype=None):
        if obj is None: return self
        res = obj.__dict__[self._name] = Debounce(partial(self.f, obj), self.wait, self.max_wait, self.leading, self.trailing)
        return res

Calling a `Debounce` is fire-and-forget. The call records its args (last call wins) and returns at once, and `f` runs later in a worker task on the event loop. The instance binds to the loop it was constructed on, or else the first loop it is called from, and keeps that loop for life. `f` can be sync or async, but a blocking `f` will stall the loop, so pass something like `partial(run_in_threadpool, f)` instead. Each call resets the `wait` timer, so a steady stream of calls fires nothing until they pause:


In [ ]:
calls = []
d = Debounce(calls.append, 0.1)
for i in range(5): d(i)
test_eq(calls, [])
await asyncio.sleep(0.25)
test_eq(calls, [4])

In [ ]:
calls.clear()
d = Debounce(calls.append, 0.1)
for i in range(4):
    d(i)
    await asyncio.sleep(0.06)   # gaps shorter than `wait`, so the timer keeps resetting
test_eq(calls, [])
await asyncio.sleep(0.15)
test_eq(calls, [3])

With `max_wait` set, a burst cannot postpone the fire forever. It comes no later than `max_wait` after the burst began.


In [ ]:
calls.clear()
d = Debounce(calls.append, 0.1, max_wait=0.2)
for i in range(8):
    d(i)
    await asyncio.sleep(0.06)  # a steady stream: plain debounce would never fire...
assert calls                   # ...but `max_wait` forced a fire mid-stream
await asyncio.sleep(0.25)
test_eq(calls[-1], 7)          # and the trailing fire still delivered the last args

In [ ]:
#| export
def debounced(wait, max_wait=None, leading=False, trailing=True):
    "Decorator: `Debounce` a function or method"
    return partial(Debounce, wait=wait, max_wait=max_wait, leading=leading, trailing=trailing)

def throttled(wait, leading=False):
    "Decorator: fire at most once per `wait` secs (`Debounce` with `max_wait=wait`)"
    return debounced(wait, max_wait=wait, leading=leading)

A throttle is a debounce whose cap equals its wait. The first call of a burst fixes the fire time, so the per-call resets never win.


In [ ]:
calls.clear()
@throttled(0.1)
def tsave(x): calls.append(x)

for i in range(5): tsave(i)
await asyncio.sleep(0.15)
test_eq(calls, [4])

On a method, `Debounce` acts as a descriptor. Each instance lazily gets its own independent debouncer, cached in the instance dict like `functools.cached_property` (so `__slots__` classes are not supported).


In [ ]:
class _Doc:
    def __init__(self): self.saved = []
    @debounced(0.1)
    def save(self, x): self.saved.append(x)

a,b = _Doc(),_Doc()
a.save(1); a.save(2); b.save(3)
await asyncio.sleep(0.15)
test_eq((a.saved,b.saved), ([2],[3]))
assert a.save is a.save

`leading=True` fires the first call of a burst right away. The trailing fire then covers the rest of the burst, and is skipped if there was no rest. `trailing=False` drops the rest instead, which gives a classic rate-limiter:


In [ ]:
calls.clear()
d = Debounce(calls.append, 0.1, leading=True)
d(1); await asyncio.sleep(0.02)
test_eq(calls, [1])         # leading fire, almost immediate
await asyncio.sleep(0.15)
test_eq(calls, [1])         # burst had no other calls, so no trailing fire
d(2); d(3); await asyncio.sleep(0.25)
test_eq(calls, [1,2,3])     # leading fired 2; trailing coalesced the rest

In [ ]:
calls.clear()
d = Debounce(calls.append, 0.1, leading=True, trailing=False)
for i in range(5): d(i)
await asyncio.sleep(0.25)
test_eq(calls, [0])         # later calls in the burst were dropped, not queued
await d.flush()
test_eq(calls, [0])         # dropped calls are gone, so there is nothing for `flush` to run

An async `f` is awaited on the loop. `flush` runs any pending call immediately, which is handy at shutdown, and `cancel` discards it. Use both from the owner loop only. Plain calls work from threads with no loop at all, and are forwarded:


In [ ]:
acalls = []
async def _asave(x): acalls.append(x)

d = Debounce(_asave, 0.1)
d(1); d(2)
await asyncio.sleep(0.15)
test_eq(acalls, [2])

In [ ]:
calls.clear()
d = Debounce(calls.append, 5)
d(1); await d.flush()
test_eq(calls, [1])          # pending call ran now, not 5s later
await d.flush()              # nothing pending: a no-op
d(2); d.cancel()
await asyncio.sleep(0.05)
test_eq(calls, [1])          # cancelled call never ran

In [ ]:
calls.clear()
d = Debounce(calls.append, 0.1)
threading.Thread(target=d, args=(7,)).start()
await asyncio.sleep(0.2)
test_eq(calls, [7])

Calls that arrive while an async `f` is still running do not overlap it. They wait for it to finish, and only the latest is kept:


In [ ]:
calls = []
started,release = asyncio.Event(),asyncio.Event()

async def _slow(x):
    calls.append(x)
    if x == 1:
        started.set()
        await release.wait()

d = Debounce(_slow, 0.05)
d(1); await started.wait()
d(2); d(3)
await asyncio.sleep(0.1)
test_eq(calls, [1])       # no overlapping call

release.set()
await asyncio.sleep(0.1)
test_eq(calls, [1,3])     # latest queued call won

A call from a thread running a different event loop is forwarded to the loop that owns the `Debounce`. The instance never migrates between loops:


In [ ]:
calls = []
owner = asyncio.get_running_loop()

async def _record(x): calls.append((x, asyncio.get_running_loop()))
d = Debounce(_record, 0.05)

async def _foreign(): d(7)
t = threading.Thread(target=lambda: asyncio.run(_foreign()))
t.start(); t.join()
await asyncio.sleep(0.1)

test_eq(calls, [(7,owner)])
assert d._loop is owner

If `f` raises, the exception ends that burst's worker task, and asyncio logs it as an unretrieved task exception. Later bursts are unaffected:

In [ ]:
calls = []
def _boom(x):
    if x==1: raise ValueError('boom')
    calls.append(x)

d = Debounce(_boom, 0.1)
d(1); await asyncio.sleep(0.15)
with expect_fail(contains='boom'): d._task.result() # the error ended up on the worker task
d(2); await asyncio.sleep(0.15)
test_eq(calls, [2])